# Lecture 11

Today's Topics:

- What is a time series?
- Time series in Altair
- Components of a time series: trend, seasonality, noise
- Additive and multiplicative models
- Time series decomposition
- Finding the dominant period with the FFT
- Stationarity and the Augmented Dickey-Fuller test
- Transforming a nonstationary series into a stationary one

In [3]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import altair as alt
import yfinance as yf
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose, STL
from statsmodels.tsa.stattools import adfuller, acf

alt.data_transformers.disable_max_rows()  # time series easily exceed Altair's 5,000-row default
np.random.seed(42)                        # reproducibility

## What is a time series?

- A time series is a sequence of data points organized in time order.
- Usually, the signal is sampled at equally spaced points in time (every second, every day, every month...).
- The series is then just the sequence of sampled values.
- We refer to this setting as *discrete* time.

![Time series example](https://raw.githubusercontent.com/sacredfuel/workshops/main/tsa-notebooks/media/TimeSeriesVisual.gif)

### What data are represented as time series?

Time series show up everywhere: finance, weather, web traffic, sensor readings, sales, and so on.

We'll start with stock prices from Yahoo Finance using the `yfinance` library.

Recent versions of `yfinance` return columns as a *MultiIndex* (one level for the price type, one for the ticker), even when you only download one ticker. Altair needs a flat table, so we write a small helper that always gives back two plain columns: `Date` and `Close`.

In [7]:
def get_close(ticker, start, end):
    """Download daily closing prices and return a flat DataFrame with Date and Close columns."""
    data = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    close = data["Close"]
    if isinstance(close, pd.DataFrame):   # newer yfinance: one column per ticker
        close = close[ticker]
    return close.rename("Close").reset_index()

googl = get_close("GOOGL", "2021-09-01", "2026-09-01")
googl.head()

,Date,Close
0,2021-09-01,143.846680
1,2021-09-02,141.936371
2,2021-09-03,142.384613
3,2021-09-07,142.920532
4,2021-09-08,142.336563


## Time series in Altair

The key to plotting time in Altair is the **temporal** data type, `:T`.

- `:T` tells Altair the field holds dates/times, so the axis gets sensible date ticks (months, years) instead of raw numbers.
- `.interactive(bind_y=False)` lets you scroll to zoom and drag to pan along the time axis only.

In [9]:
alt.Chart(googl).mark_line().encode(
    x=alt.X("Date:T", title="Date"),
    y=alt.Y("Close:Q", title="Price (USD)"),
    tooltip=[alt.Tooltip("Date:T"), alt.Tooltip("Close:Q", format="$.2f")],
).properties(
    width=700, height=300, title="Google Stock Price vs Time"
).interactive(bind_y=False)

alt.Chart(...)

### Time units

Altair can also *aggregate* by calendar units directly in the encoding, using a **time unit** such as `year`, `yearmonth`, `month`, or `day`.

- `yearmonth(Date)` groups all the days of each month together.
- `mean(Close)` averages within each group.

This is the same as `googl.resample("M", on="Date").mean()` in pandas, but done inside the chart.

In [10]:
alt.Chart(googl).mark_line(point=True).encode(
    x=alt.X("yearmonth(Date):T", title="Month"),
    y=alt.Y("mean(Close):Q", title="Average monthly price (USD)"),
    tooltip=[alt.Tooltip("yearmonth(Date):T", title="Month"),
             alt.Tooltip("mean(Close):Q", format="$.2f", title="Average close")],
).properties(width=700, height=300, title="GOOGL: monthly average closing price")

alt.Chart(...)

Not every time series comes with real dates. The Lake Huron dataset stores the year as a plain number (`1875.0`, `1876.0`, ...).

If we plot it as `:Q`, Altair formats the axis like any other number (`1,880`). Converting the year to a real date and using `:T` gives a proper time axis.

In [11]:
huron = sm.datasets.get_rdataset("LakeHuron", "datasets").data
huron["Year"] = pd.to_datetime(huron["time"].astype(int).astype(str), format="%Y")

alt.Chart(huron).mark_line().encode(
    x=alt.X("Year:T", title="Year"),
    y=alt.Y("value:Q", title="Elevation (feet)", scale=alt.Scale(zero=False)),
    tooltip=[alt.Tooltip("Year:T", format="%Y"), alt.Tooltip("value:Q", format=".2f")],
).properties(width=700, height=300, title="Lake Huron Water Level (1875-1972)")

alt.Chart(...)

### Time series analysis

The main objectives of time series analysis are:

1. To find a **model** that adequately explains the **dependence** observed in a time series.
2. To **predict** or **forecast** future values of the time series based on those we have already observed.
3. To **adjust** our observed data into a form more suited for answering the question of interest.

### Applications

**Economic forecasting**

- Time series analysis is used in macroeconomic predictions.
- The World Trade Organization uses time series forecasting to predict levels of international trade.
- The Federal Reserve uses time series forecasts of the economy to set interest rates [[source](https://www.federalreserve.gov/pubs/feds/2009/200910/200910pap.pdf)].

**Demand forecasting**

- Time series analysis is used to predict demand at different levels of granularity.
- Amazon and other e-commerce companies use time series models to predict demand at a product and geography level [[source](https://www.theverge.com/2014/1/18/5320636/amazon-plans-to-ship-your-packages-before-you-even-buy-them)].
- This helps meet customer needs (fast shipping) and reduce inventory waste.

**Anomaly detection**

- Used to detect anomalous behaviour in the underlying system by looking for unusual patterns in the time series.
- Widely used in manufacturing to detect defects and target preventive maintenance.
- With IoT devices, anomaly detection is used in machinery-heavy industries such as petroleum and gas [[source](https://arxiv.org/abs/1607.02480)].

## Time series components

A time series is often assumed to be composed of three components:

- **Trend**: the long-term direction of the data.
- **Seasonality**: the part that repeats itself at a fixed interval.
- **Noise**: whatever is left that is neither trend nor seasonality.

### Trend

- Kendall writes that "the essential idea of trend is that it shall be smooth."
- Trend captures the general direction of the time series.
- Trend can be increasing, decreasing, or constant.
- It can increase/decrease in different ways over time (linearly, exponentially, etc).
- When the trend is removed, we obtain a **detrended** series.

Altair can fit a trend line for us with `transform_regression`. Here we layer the price line and a dashed linear trend.

In [13]:
base = alt.Chart(googl).encode(
    x=alt.X("Date:T", title="Date"),
    y=alt.Y("Close:Q", title="Price (USD)"),
)

price = base.mark_line().encode(
    tooltip=[alt.Tooltip("Date:T"), alt.Tooltip("Close:Q", format="$.2f")]
)
trendline = base.transform_regression("Date", "Close").mark_line(
    color="firebrick", strokeDash=[6, 4], strokeWidth=2
)

(price + trendline).properties(
    width=700, height=300, title="Google Stock Price with Linear Trendline (2021-2026)"
)

alt.LayerChart(...)

`transform_regression` also accepts `method="poly"` with an `order`, or `method="exp"`, `"log"`, `"pow"` for non-linear trends.

Next, let's build each component from scratch so we know exactly what's in the data. We'll simulate 5 years of trading days (252 per year) and give the data real business-day dates.

In [14]:
dates = pd.bdate_range("2021-09-01", periods=252 * 5)   # 5 years of trading days
t = np.arange(len(dates))

trend = 75 + 0.07 * t                                   # rises steadily
seasonal = 4 * np.sin(2 * np.pi * t / 252)              # repeats every 252 trading days
noise = np.random.normal(loc=0, scale=1, size=len(t))   # random fluctuations

parts = pd.DataFrame({"Date": dates, "Trend": trend, "Seasonality": seasonal, "Noise": noise})
parts.head()

,Date,Trend,Seasonality,Noise
0,2021-09-01,75.00,0.000000,0.496714
1,2021-09-02,75.07,0.099723,-0.138264
2,2021-09-03,75.14,0.199384,0.647689
3,2021-09-06,75.21,0.298920,1.523030
4,2021-09-07,75.28,0.398271,-0.234153


### Seasonality

- Periodic fluctuations that occur at regular intervals.
- Consistent and predictable patterns over a specific period (daily, monthly, quarterly, yearly).
- When we remove a seasonal component we say that we **deseasonalize** the series.

Seasonality can be driven by many factors:

- Naturally occurring events, such as weather changing with the time of year.
- Business or administrative procedures, such as the start and end of a school year.
- Social or cultural behaviour, such as holidays or religious observances.

The period of `np.sin(2 * np.pi * t / P)` is exactly `P` steps, so our seasonal component repeats every 252 trading days (once a year).

### Noise

- Noise is the random fluctuation left over after trend and seasonality are removed.
- One should not see a trend or seasonal pattern in the noise.
- It represents short-term, unpredictable fluctuations.

To plot all three components we need the data in **long form** (one row per date per component). Altair can do this reshaping itself with `transform_fold`, which is the chart-side version of `pd.melt`.

Then `row=` splits the components into stacked panels, and `resolve_scale(y="independent")` gives each panel its own y-axis, since the components have very different ranges.

In [15]:
alt.Chart(parts).transform_fold(
    ["Trend", "Seasonality", "Noise"], as_=["component", "value"]
).mark_line(strokeWidth=1.5).encode(
    x=alt.X("Date:T", title="Date"),
    y=alt.Y("value:Q", title=None, scale=alt.Scale(zero=False)),
    color=alt.Color("component:N", legend=None),
    row=alt.Row("component:N", sort=["Trend", "Seasonality", "Noise"], title=None),
).properties(width=700, height=110).resolve_scale(y="independent")

alt.Chart(...)

## Decomposition models

A time series can be modeled as a combination of its components in two main ways.

### Additive model

The observed series is the **sum** of its components:

$$X(t) = T(t) + S(t) + N(t)$$

where

- $X(t)$ is the time series
- $T(t)$ is the trend
- $S(t)$ is the seasonality
- $N(t)$ is the noise

Additive models are used when the size of the seasonal swings and the noise **does not depend** on the level of the trend.

In [16]:
additive = trend + seasonal + noise

additive_df = pd.DataFrame({"Date": dates, "Value": additive})

alt.Chart(additive_df).mark_line().encode(
    x=alt.X("Date:T", title="Date"),
    y=alt.Y("Value:Q", title="Price (USD)", scale=alt.Scale(zero=False)),
    tooltip=[alt.Tooltip("Date:T"), alt.Tooltip("Value:Q", format=".2f")],
).properties(width=700, height=250, title="Additive Time Series")

alt.Chart(...)

### Multiplicative model

The observed series is the **product** of its components:

$$X(t) = T(t) \cdot S(t) \cdot N(t)$$

- Here $S(t)$ and $N(t)$ are *factors* centered around 1. For example, $S(t) = 1.15$ means "15% above the trend."
- Multiplicative models are used when the seasonal swings **grow with** the level of the trend.
- Taking the log turns a multiplicative model into an additive one:

$$\log X(t) = \log T(t) + \log S(t) + \log N(t)$$

This only works if every value is positive, which is why the seasonal and noise factors must stay above 0.

In [17]:
seasonal_factor = 1 + 0.15 * np.sin(2 * np.pi * t / 252)    # swings +/-15% around the trend
noise_factor = 1 + np.random.normal(0, 0.01, len(t))         # about +/-1% noise
multiplicative = trend * seasonal_factor * noise_factor

mult_df = pd.DataFrame({"Date": dates, "Value": multiplicative, "Log value": np.log(multiplicative)})

mult_chart = alt.Chart(mult_df).mark_line().encode(
    x=alt.X("Date:T", title="Date"),
    y=alt.Y("Value:Q", title="Price (USD)", scale=alt.Scale(zero=False)),
).properties(width=340, height=250, title="Multiplicative: swings grow with the trend")

log_chart = alt.Chart(mult_df).mark_line(color="darkorange").encode(
    x=alt.X("Date:T", title="Date"),
    y=alt.Y("Log value:Q", title="log(Price)", scale=alt.Scale(zero=False)),
).properties(width=340, height=250, title="After log: swings are constant")

mult_chart | log_chart

alt.HConcatChart(...)

The `|` operator places charts side by side (`alt.hconcat`), and `&` stacks them (`alt.vconcat`).

Notice that on the left the seasonal swings get larger as the price rises, while on the right (after taking the log) the swings stay the same size. For the rest of this lecture we focus on the additive model.

## Time series decomposition

Now let's go the other way: we have the combined series, and we want to recover the components.

A simple and often useful first step is to estimate a **linear** trend with a 1st-order polynomial fit, then subtract it to get a **detrended** series.

In [18]:
slope, intercept = np.polyfit(t, additive, 1)   # estimate the line's coefficients
linear_trend = intercept + slope * t             # the fitted linear trend
detrended = additive - linear_trend              # remove the trend

detrend_df = pd.DataFrame({
    "Date": dates,
    "Original": additive,
    "Linear trend": linear_trend,
    "Detrended": detrended,
})

alt.Chart(detrend_df).transform_fold(
    ["Original", "Linear trend", "Detrended"], as_=["series", "value"]
).mark_line(strokeWidth=1.5).encode(
    x=alt.X("Date:T", title="Date"),
    y=alt.Y("value:Q", title="Value"),
    color=alt.Color("series:N", sort=["Original", "Linear trend", "Detrended"], title=None),
    tooltip=[alt.Tooltip("Date:T"), "series:N", alt.Tooltip("value:Q", format=".2f")],
).properties(width=700, height=250, title="Removing a linear trend")

alt.Chart(...)

Notice we stored the fitted line as `linear_trend` instead of overwriting `trend`. We still want the true trend around to compare against.

### `seasonal_decompose`

Next we use `seasonal_decompose` from `statsmodels` ([docs](https://www.statsmodels.org/stable/generated/statsmodels.tsa.seasonal.seasonal_decompose.html)) to isolate all three components.

- We have to specify the type of model (`"additive"` or `"multiplicative"`) and the main `period`.
- Since the market is open about 252 days a year, we use `period=252`.

In [19]:
additive_decomposition = seasonal_decompose(additive, model="additive", period=252)

To plot the decomposition we'll write a reusable helper. It builds four stacked panels (original, trend, seasonality, residuals) and adds an **interval selection** (a brush) to the top panel.

- `alt.selection_interval(encodings=["x"])` lets you click and drag along the x-axis to select a time window.
- The other three panels use that selection as their x-axis domain (`scale(domain=brush)`), so dragging on the top panel **zooms all the panels below it** to the same window.

In [20]:
def decomposition_chart(x, original, result, title):
    """Stacked original / trend / seasonal / residual panels, linked by a brush on the top panel."""
    df = pd.DataFrame({
        "x": x,
        "Original": original,
        "Trend": result.trend,
        "Seasonality": result.seasonal,
        "Residuals": result.resid,
    })
    x_type = "T" if pd.api.types.is_datetime64_any_dtype(df["x"]) else "Q"
    brush = alt.selection_interval(encodings=["x"])

    def panel(column, zoomable):
        x_enc = alt.X(f"x:{x_type}", title=None)
        if zoomable:
            x_enc = x_enc.scale(domain=brush)
        return alt.Chart(df).mark_line(strokeWidth=1.5, clip=True).encode(
            x=x_enc,
            y=alt.Y(f"{column}:Q", scale=alt.Scale(zero=False)),
            tooltip=[alt.Tooltip(f"x:{x_type}", title="Time"),
                     alt.Tooltip(f"{column}:Q", format=".2f")],
        ).properties(width=700, height=110)

    top = panel("Original", zoomable=False).add_params(brush).properties(
        title=f"{title} (drag on this panel to zoom the others)"
    )
    return alt.vconcat(top, panel("Trend", True), panel("Seasonality", True), panel("Residuals", True))

decomposition_chart(dates, additive, additive_decomposition, "seasonal_decompose")

alt.VConcatChart(...)

Notice the trend and residuals are **missing for the first and last 126 days** (half of 252).

- `seasonal_decompose` estimates the trend with a *centered moving average* over one full period.
- A centered window of 252 needs 126 values on each side, so it can't be computed at the edges.
- Altair simply leaves gaps where the values are missing (`NaN`).

Also notice the seasonal component is a bit noisy. `seasonal_decompose` estimates the seasonal value for each day of the cycle by averaging that day across all cycles. We only have 5 cycles (5 years), so each estimate is an average of just 5 noisy values.

### STL: Seasonal and Trend decomposition using LOESS

Next we try a second method, `STL`, which estimates each component with LOESS (locally weighted regression) instead of a plain moving average. It has no gaps at the edges, and `robust=True` reduces the influence of outliers.

- STL smooths each day-of-cycle's values across the years (all the first days, all the second days, ...).
- With only 5 years, fitting a *line* through 5 points per day overfits at the edges, so we set `seasonal_deg=0` to fit a *constant* (a local average) instead.

In [21]:
stl_decomposition = STL(additive, period=252, seasonal_deg=0, robust=True).fit()
decomposition_chart(dates, additive, stl_decomposition, "STL")

alt.VConcatChart(...)

### Which method to use?

Use `seasonal_decompose` when:

- Your time series has a clear and stable seasonal pattern and trend.
- You prefer a simpler model with fewer parameters to adjust.
- The seasonal amplitude is constant over time.

Use `STL` when:

- Your time series has seasonality that may change over time.
- You need to handle outliers without them distorting the trend and seasonal components.
- You are dealing with non-linear trends and need more control over the decomposition.

## Identifying the dominant period

`seasonal_decompose` and `STL` both need the period as a parameter. Here we knew it was 252, but we won't always be so lucky. How can we find it?

- Plot the data and count how many steps it takes for the cycle to repeat.
- Use the **Fast Fourier Transform** (FFT) on the detrended signal.

The FFT rewrites a signal as a sum of sine waves of different frequencies. The **magnitude** at each frequency tells us how strongly that sine wave is present. The frequency with the largest magnitude is the dominant cycle, and its period is `1 / frequency`.

We detrend first because a trend looks like a very slow wave and would otherwise dominate the spectrum.

In [22]:
def fft_analysis(signal):
    """Return the dominant period of a signal and a DataFrame of its frequency spectrum."""
    steps = np.arange(len(signal))
    slope, intercept = np.polyfit(steps, signal, 1)
    detrended = signal - (intercept + slope * steps)          # remove the linear trend

    magnitudes = np.abs(np.fft.rfft(detrended))               # strength of each frequency
    frequencies = np.fft.rfftfreq(len(signal))                # in cycles per time step
    magnitudes, frequencies = magnitudes[1:], frequencies[1:] # drop frequency 0 (the mean)

    dominant_frequency = frequencies[np.argmax(magnitudes)]
    dominant_period = 1 / dominant_frequency
    print(f"Dominant frequency: {dominant_frequency:.5f} cycles per step")
    print(f"Dominant period:    {dominant_period:.1f} steps")

    spectrum = pd.DataFrame({"frequency": frequencies, "period": 1 / frequencies, "magnitude": magnitudes})
    return dominant_period, spectrum

period, spectrum = fft_analysis(additive)

Dominant frequency: 0.00397 cycles per step
Dominant period:    252.0 steps


We plot the spectrum as a **stem plot**: a thin rule from zero up to each magnitude, with a point on top. We only show the low frequencies, where the interesting peaks are. Hover over a stem to see the period it corresponds to.

In [23]:
low = spectrum[spectrum.frequency < 0.02]

stems = alt.Chart(low).mark_rule().encode(
    x=alt.X("frequency:Q", title="Frequency (cycles per trading day)"),
    y=alt.Y("magnitude:Q", title="Magnitude"),
)
tips = alt.Chart(low).mark_point(filled=True, size=40).encode(
    x="frequency:Q",
    y="magnitude:Q",
    tooltip=[alt.Tooltip("frequency:Q", format=".5f"),
             alt.Tooltip("period:Q", format=".1f", title="period (days)"),
             alt.Tooltip("magnitude:Q", format=".0f")],
)

(stems + tips).properties(width=700, height=250, title="Frequency Spectrum")

alt.LayerChart(...)

The FFT found the 252-day cycle even though the series also contains noise and a trend.

One caveat: the FFT can only report periods of the form `N / k`, where `N` is the length of the series and `k` is a whole number. Our series has $N = 1260 = 5 \times 252$, so 252 is exactly one of the possible periods. With other lengths, the answer will be close but not exact, so round it to a sensible value (like 7, 12, 24, or 252) based on what you know about the data.

Let's try a series that repeats every 24 hours, sampled hourly for 20 days:

In [24]:
hours = np.arange(24 * 20)
hourly_seasonal = 12 + np.sin(2 * np.pi * hours / 24)

fft_analysis(hourly_seasonal);

Dominant frequency: 0.04167 cycles per step
Dominant period:    24.0 steps


---

# Stationarity

### Why do we care about stationarity?

- Stationarity is a key assumption in time series analysis.
- Without it, many basic time series models break.
- Transformations can be applied to convert a nonstationary time series into a stationary one.

### What is stationarity?

Intuition: a time series is stationary when **the process that generates it doesn't change over time**. In practice we check that the series has:

- **Constant mean**: no trend, and no periodic swings in the level.
- **Constant variance**: the spread doesn't grow or shrink over time.
- **Constant autocorrelation structure**: the correlation between $X(t)$ and $X(t-k)$ depends only on the lag $k$, not on *when* you look.

A seasonal pattern breaks the first condition, since the mean shifts up and down with the cycle.

We'll make a lot of single-series line plots, so let's write a small helper. It returns an Altair chart, so we can keep layering things on top of it with `+`.

In [25]:
def run_sequence_plot(x, y, title, xlabel="Time", ylabel="Values"):
    df = pd.DataFrame({"x": x, "y": y})
    return alt.Chart(df).mark_line(color="black", strokeWidth=1.2).encode(
        x=alt.X("x:Q", title=xlabel),
        y=alt.Y("y:Q", title=ylabel, scale=alt.Scale(zero=False)),
        tooltip=[alt.Tooltip("x:Q", title=xlabel), alt.Tooltip("y:Q", format=".2f", title=ylabel)],
    ).properties(width=700, height=200, title=title)


def hline(y, color="firebrick"):
    """A horizontal reference line at height y."""
    return alt.Chart(pd.DataFrame({"y": [y]})).mark_rule(color=color, strokeWidth=2).encode(y="y:Q")


def hband(low, high, color="firebrick"):
    """A shaded horizontal band between two heights."""
    return alt.Chart(pd.DataFrame({"low": [low], "high": [high]})).mark_rect(
        color=color, opacity=0.15
    ).encode(y="low:Q", y2="high:Q")

#### White noise

- We create a variable `time` with equally-spaced time steps.
- Then we sample values from a Normal distribution with mean 0 and standard deviation 1.
- This is **white noise**: every value is drawn independently from the same distribution, so it's stationary.

The red line is the mean, and the shaded band is mean ± 1.96 standard deviations (where about 95% of the values should fall).

In [26]:
T = 200
time = np.arange(T)
white_noise = np.random.normal(loc=0, scale=1.0, size=T)

mean, std = white_noise.mean(), white_noise.std()

hband(mean - 1.96 * std, mean + 1.96 * std) + run_sequence_plot(time, white_noise, "White noise") + hline(mean)

alt.LayerChart(...)

The mean and the spread look the same everywhere, and there's no periodic pattern. What about autocorrelation?

#### Autocorrelation

- Autocorrelation measures how much the current value is correlated with a past value of the same series.
- For example, today's temperature is usually highly correlated with yesterday's.
- To talk about this we need the idea of a **lag**, i.e., a delay.

![Autocorrelation](https://raw.githubusercontent.com/sacredfuel/workshops/main/tsa-notebooks/media/Autocorrelation.gif)

- Say you want to know if today's temperature correlates more with yesterday's or with the temperature two days ago.
- You compute the correlation between the series and a copy of itself shifted by 1 step (lag 1), then by 2 steps (lag 2), and so on.
- Autocorrelation often shows up as a visible pattern, whereas a series without autocorrelation looks random.

**Constant autocorrelation**: in a place with a very stable climate, the correlation between today's and tomorrow's temperature is about the same as between the temperature 30 days ago and 29 days ago.

**Time-varying autocorrelation**: stock prices may follow a steady pattern during a calm market, but the pattern can change dramatically during a crash or a major company event.

#### Random walk

The random walk is one of the most important **nonstationary** time series. Each value is the previous value plus a random step:

$$X(t) = X(t-1) + \epsilon_t$$

where the steps $\epsilon_t$ are drawn independently (white noise). Adding up the steps is exactly what `np.cumsum` does.

In [27]:
random_walk = 4 + np.cumsum(np.random.normal(loc=0, scale=2.5, size=T))

run_sequence_plot(time, random_walk, "Random walk")

alt.Chart(...)

- The random walk looks smooth from one step to the next, because each value is the previous value plus a small step.
- But over time it **wanders far** from where it started. Its variance grows with time ($\text{Var}[X(t)] = \sigma^2 t$), so it is nonstationary.

We can *see* the difference in autocorrelation with a **lag plot**: a scatter plot of each value $X(t)$ against the previous value $X(t-1)$.

In [28]:
def lag_plot(series, title, lag=1):
    df = pd.DataFrame({"previous": series[:-lag], "current": series[lag:]})
    return alt.Chart(df).mark_circle(size=30, opacity=0.6).encode(
        x=alt.X("previous:Q", title=f"X(t-{lag})", scale=alt.Scale(zero=False)),
        y=alt.Y("current:Q", title="X(t)", scale=alt.Scale(zero=False)),
        tooltip=[alt.Tooltip("previous:Q", format=".2f"), alt.Tooltip("current:Q", format=".2f")],
    ).properties(width=320, height=320, title=title)

lag_plot(white_noise, "White noise: no relationship") | lag_plot(random_walk, "Random walk: strong relationship")

alt.HConcatChart(...)

The **autocorrelation function** (ACF) does this for every lag at once. Each bar is the correlation between the series and itself shifted by that many steps. The shaded band marks roughly where correlations are too small to be distinguishable from zero.

In [29]:
def acf_chart(series, title, nlags=30):
    df = pd.DataFrame({"lag": np.arange(nlags + 1), "acf": acf(series, nlags=nlags)})
    bound = 1.96 / np.sqrt(len(series))
    bars = alt.Chart(df).mark_bar(width=5).encode(
        x=alt.X("lag:Q", title="Lag", scale=alt.Scale(domain=[-1, nlags + 1], nice=False)),
        y=alt.Y("acf:Q", title="Autocorrelation", scale=alt.Scale(domain=[-1, 1])),
        tooltip=["lag:Q", alt.Tooltip("acf:Q", format=".2f")],
    )
    return (hband(-bound, bound, color="steelblue") + bars).properties(width=320, height=220, title=title)

acf_chart(white_noise, "ACF: white noise") | acf_chart(random_walk, "ACF: random walk")

alt.HConcatChart(...)

- White noise: apart from lag 0 (a series is always perfectly correlated with itself), the bars stay inside the band.
- Random walk: the correlations are high and fade out very slowly. That slow decay is a classic sign of nonstationarity.

#### Time series with trend

In [30]:
trended = 2.75 * time + white_noise

run_sequence_plot(time, trended, "Time series with trend")

alt.Chart(...)

Is this series stationary?

No. The mean changes over time (there is a trend).

#### Time series with seasonality

In [31]:
seasonality = 10 + 10 * np.sin(2 * np.pi * time / 24)   # period of 24 steps

run_sequence_plot(time, seasonality, "Time series with seasonality")

alt.Chart(...)

Is this series stationary?

- The mean of a periodic signal over a **complete cycle** is always the same (10 here).
- But over half a cycle, or any window that isn't a whole number of cycles, the mean changes.
- So the mean is not constant over time, and the series is not stationary.

![Sin stationary](https://raw.githubusercontent.com/sacredfuel/workshops/main/tsa-notebooks/media/SinStationary.gif)

We can show this directly in Altair with `transform_window`, which computes a **rolling** statistic over a sliding window of rows.

- `frame=[-11, 0]` means "this row and the 11 before it", a window of 12 steps (half a cycle).
- `frame=[-23, 0]` is a window of 24 steps (a full cycle).

The half-cycle rolling mean swings up and down, while the full-cycle rolling mean is flat at 10. (We'll use `transform_window` a lot more next lecture.)

At the very start, the window isn't full yet (there are fewer than 24 earlier values), so Altair averages whatever rows are available. That's why the blue line only settles at 10 after the first cycle.

In [32]:
season_df = pd.DataFrame({"t": time, "value": seasonality})

season_base = alt.Chart(season_df).encode(x=alt.X("t:Q", title="Time"))

season_raw = season_base.mark_line(color="lightgray").encode(y=alt.Y("value:Q", title="Values"))

half_cycle = season_base.transform_window(
    rolling_mean="mean(value)", frame=[-11, 0]
).mark_line(color="firebrick", strokeWidth=2).encode(y="rolling_mean:Q")

full_cycle = season_base.transform_window(
    rolling_mean="mean(value)", frame=[-23, 0]
).mark_line(color="steelblue", strokeWidth=2).encode(y="rolling_mean:Q")

(season_raw + half_cycle + full_cycle).properties(
    width=700, height=250,
    title="Rolling mean over half a cycle (red) vs a full cycle (blue)",
)

alt.LayerChart(...)

## Detecting stationarity

Plotting the series is always the first step. Beyond that, we can use a statistical test. We'll focus on the **Augmented Dickey-Fuller (ADF) test**.

### Augmented Dickey-Fuller test

- **Null hypothesis** $H_0$: the series is nonstationary (it has a *unit root*, like a random walk).
- **Alternative hypothesis** $H_A$: the series is stationary.

As with any statistical test, we pick a significance level, commonly $\alpha = 0.05$ or $\alpha = 0.01$.

- $p = 0.0001 < \alpha$: **reject** $H_0$. We have evidence the series is stationary.
- $p = 0.43 > \alpha$: **fail to reject** $H_0$. We do *not* have enough evidence to call it stationary.

Note the wording in the second case. Failing to reject $H_0$ doesn't prove the series is nonstationary; it just means the test couldn't rule it out.

#### ADF on white noise

In [33]:
adf, pvalue, usedlag, nobs, critical_values, icbest = adfuller(white_noise)

print(f"ADF statistic: {adf:.2f}")
print(f"p-value: {pvalue:.6f}")
print(f"nobs: {nobs}")
print(f"critical values: { {level: round(float(value), 2) for level, value in critical_values.items()} }")

ADF statistic: -14.45
p-value: 0.000000
nobs: 199
critical values: {'1%': -3.46, '5%': -2.88, '10%': -2.57}


- `adf` is the test statistic. The **more negative** it is, the stronger the evidence that the series is stationary.
- `pvalue` is very close to zero, so we **reject** $H_0$ and conclude the series is stationary.
- `nobs` is the number of observations used in the test.
- `critical_values` are the thresholds for common significance levels. A statistic below the 5% value (about -2.88) is enough to reject $H_0$ at the 5% level.

See the [statsmodels docs](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.adfuller.html) for `usedlag` and `icbest`.

Since we'll run this test several times, let's wrap it in a function.

In [34]:
def adf_test(series, regression="c"):
    adf, pvalue, _, _, critical_values, _ = adfuller(series, regression=regression)
    verdict = "reject H0: stationary" if pvalue < 0.05 else "fail to reject H0: no evidence of stationarity"
    print(f"ADF statistic: {adf:.2f}   p-value: {pvalue:.4f}   5% critical value: {critical_values['5%']:.2f}")
    print(f"-> {verdict}")

#### ADF on a series with trend

The `regression` argument controls what the test allows the stationary series to look like:

- `regression="c"` (default): stationary around a **constant** mean.
- `regression="ct"`: stationary around a **constant plus a linear trend**.

In [35]:
print("regression='c'")
adf_test(trended, regression="c")
print("\nregression='ct'")
adf_test(trended, regression="ct")

regression='c'
ADF statistic: 0.18   p-value: 0.9712   5% critical value: -2.88
-> fail to reject H0: no evidence of stationarity

regression='ct'
ADF statistic: -14.42   p-value: 0.0000   5% critical value: -3.43
-> reject H0: stationary


The two settings give opposite answers, and both are correct for the question they ask.

- With `"c"`, the series is clearly not stationary around a constant, because the mean keeps rising.
- With `"ct"`, the test finds that once a straight-line trend is accounted for, what's left is stationary.

A series like this is called **trend-stationary**: removing the deterministic trend makes it stationary.

#### ADF on the random walk

In [36]:
print("regression='c'")
adf_test(random_walk, regression="c")
print("\nregression='ct'")
adf_test(random_walk, regression="ct")

regression='c'
ADF statistic: -0.95   p-value: 0.7722   5% critical value: -2.88
-> fail to reject H0: no evidence of stationarity

regression='ct'
ADF statistic: -2.63   p-value: 0.2659   5% critical value: -3.43
-> fail to reject H0: no evidence of stationarity


The random walk fails both versions. Subtracting a straight line doesn't help, because its wandering isn't a fixed trend: it's a sum of random steps (a *stochastic* trend). To make it stationary we need **differencing**, which we'll see below.

## Common transformations to achieve stationarity

Transformation | Fixes
:---|:---
Subtract an estimated trend | Non-constant mean from a deterministic trend
Apply a log | Variance that grows with the level (multiplicative series)
Differencing, $X(t) - X(t-1)$ | A stochastic trend (unit root), like a random walk
Seasonal differencing, $X(t) - X(t-L)$ | A periodic component with period $L$

- Sometimes you'll need several transformations on the same dataset.
- This is a small subset of the possible transformations, but it covers most cases you'll meet.

### Subtracting the estimated trend and seasonality

In [37]:
trend_seasonality = trended + seasonality

run_sequence_plot(time, trend_seasonality, "Time series with trend and seasonality")

alt.Chart(...)

In [38]:
adf_test(trend_seasonality)

ADF statistic: 0.20   p-value: 0.9725   5% critical value: -2.88
-> fail to reject H0: no evidence of stationarity


Nonstationary, as expected. Next we decompose it with `period=24`, reusing our `decomposition_chart` helper. (It works with plain numbers on the x-axis too.)

In [39]:
ts_decomposition = seasonal_decompose(trend_seasonality, model="additive", period=24)

decomposition_chart(time, trend_seasonality, ts_decomposition, "Trend + seasonality, period 24")

alt.VConcatChart(...)

The residuals are what's left after removing the estimated trend and seasonality.

- The first and last 12 residuals are `NaN`, for the same reason as before: the trend comes from a centered moving average with a window of 24, which needs 12 values on each side.
- We drop the `NaN`s before running the test.

In [40]:
residuals = ts_decomposition.resid
residuals = residuals[~np.isnan(residuals)]

adf_test(residuals)

ADF statistic: -6.58   p-value: 0.0000   5% critical value: -2.88
-> reject H0: stationary


After removing trend and seasonality, the residuals are stationary.

### Seasonal differencing

- For a series with seasonality of period $L$, there's a strong correlation between $X(t)$ and $X(t-L)$.
- **Seasonal differencing** subtracts the value from one period earlier:

$$X'(t) = X(t) - X(t-L)$$

- The result is $L$ steps shorter, since the first $L$ values have nothing to subtract.

![Seasonal Differencing](https://raw.githubusercontent.com/sacredfuel/workshops/main/tsa-notebooks/media/SeasonalDifferencing.gif)

In [41]:
L = 24
seasonal_diff = trend_seasonality[L:] - trend_seasonality[:-L]   # X(t) - X(t-L)

run_sequence_plot(time[L:], seasonal_diff, "After seasonal differencing (L = 24)") + hline(seasonal_diff.mean())

alt.LayerChart(...)

In [42]:
adf_test(seasonal_diff)

ADF statistic: -4.71   p-value: 0.0001   5% critical value: -2.88
-> reject H0: stationary


The seasonality is gone, and so is the trend. Why?

- The trend rises by 2.75 per step, so over 24 steps it rises by $2.75 \times 24 = 66$.
- After seasonal differencing, the trend turns into a **constant** of about 66 (the red line), which is a constant mean.
- If the trend were curved instead of straight, we'd follow up with a regular first difference to remove what's left.

### Differencing a random walk

First differencing undoes exactly what a random walk is built from:

$$X(t) - X(t-1) = \epsilon_t$$

so differencing a random walk gives back the white-noise steps. `np.diff` computes this.

In [43]:
rw_diff = np.diff(random_walk)

run_sequence_plot(time[1:], rw_diff, "Random walk after first differencing").properties(width=360) | acf_chart(rw_diff, "ACF after differencing")

alt.HConcatChart(...)

In [44]:
adf_test(rw_diff)

ADF statistic: -8.91   p-value: 0.0000   5% critical value: -2.88
-> reject H0: stationary
